# DemandForest: New Product Forecasting Pipeline
**Reference:** *Forecasting demand profiles of new products* (van Steenbergen & Mes, 2020)

## Overview
This notebook implements the **DemandForest** methodology to generate pre-launch forecasts for new products without historical sales data. The pipeline is particularly suited for B2B distribution environments (like dental consumables and tools) where quantifying demand uncertainty is just as critical as point accuracy for inventory management.

**Methodology Steps:**
1. **Profile Extraction:** Historical demands of existing products are normalized to extract their underlying demand profiles $p_x$ over a fixed introduction horizon (e.g., $T=8$ or $18$ weeks).
2. **Clustering:** K-Means clusters these normalized profiles. The optimal number of clusters ($k$) is selected via a majority vote among the Davies-Bouldin Index, Silhouette Coefficient, and Calinski-Harabasz Index.
3. **Profile Classification:** A Random Forest (RF) classifier is trained to map launch-time product characteristics $f_z$ to the assigned demand profile.
4. **Total Demand Regression:** A Quantile Regression Forest (QRF) is trained on product characteristics to predict the total aggregate demand $d^h_x$ and conditional quantiles (e.g., 5th and 95th percentiles).
5. **Synthesis:** For new products $y$, the predicted profile $\hat{p}_y$ and the predicted total demand $\hat{d}^h_y$ are multiplied to generate the final time-series forecast $\hat{d}_{y,t}$.

*Note: Requires the `quantile-forest` package for QRF implementation.*

In [ ]:
# Install Quantile Regression Forest package (if not installed)
!pip install quantile-forest -q

# DATA HANDLING & MATH
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.stats as stats # NEW: Added for theoretical distribution fitting

# MACHINE LEARNING & EVALUATION
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from quantile_forest import RandomForestQuantileRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    accuracy_score, cohen_kappa_score, mean_squared_error, make_scorer
)
from sklearn.inspection import permutation_importance # NEW: For feature importance

# DATETIME & VISUALIZATION
import time
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# PARALLEL
from joblib import Parallel, delayed
import multiprocessing
import warnings
warnings.filterwarnings('ignore')
import os
import requests
from tqdm import tqdm

# Google Trends
!pip install pytrends
# !pip install urllib3==1.26.15
# import urllib3
# print(f"urllib3 version: {urllib3.__version__}")
# # This MUST print 1.26.15. If it prints 2.x.x, the kernel didn't restart properly.
import random
from pytrends.request import TrendReq

---
## 1. Environment & Path Configuration

In [ ]:
# Mount Google Drive
# drive.mount('/content/drive')

# # Define paths (Adjust to your B2B data directory)
# folder_path = Path('/content/drive/MyDrive/input_folder')
# out_folder  = Path('/content/drive/MyDrive/output_folder')
# out_folder.mkdir(parents=True, exist_ok=True)

# print(f'Input folder:  {folder_path}')
# print(f'Output folder: {out_folder}')

# Global Pipeline Parameters
test_share = 0.2
val_share_of_train = 0.2
FORECAST_HORIZON_WEEKS = 12
N_TREES = 2000              # Recommended tree count for RF/QRF stability
RANDOM_STATE = 42

---
## 2. Profile Extraction & Data Preparation
To cluster demand patterns, we first isolate the historical total demand $d^h_x$ and the normalized demand profile $p_x = (p_{x,t})$ where $p_{x,t} = d_{x,t} / d^h_x$.

In [ ]:
def extract_demand_profiles(df, id_col='product_code_base', time_col='week_num', qty_col='item_quantity'):
    """
    Extracts the total demand and normalized weekly demand profiles.
    """
    total_demand = df.groupby(id_col)[qty_col].sum().reset_index()
    total_demand.rename(columns={qty_col: 'total_demand_h'}, inplace=True)

    valid_products = total_demand[total_demand['total_demand_h'] > 0]
    df_valid = df[df[id_col].isin(valid_products[id_col])].copy()

    pivot_df = df_valid.pivot_table(index=id_col, columns=time_col, values=qty_col, aggfunc='sum').fillna(0)

    available_weeks = [w for w in range(1, FORECAST_HORIZON_WEEKS + 1) if w in pivot_df.columns]
    pivot_df = pivot_df[available_weeks]

    # Normalize to create the shape profile (px)
    profiles = pivot_df.div(pivot_df.sum(axis=1), axis=0)

    return profiles, total_demand

In [ ]:
def fetch_historical_trend_matrix(unique_brands, anchor_term="dental products", timeframe='today 5-y'):
    """
    Fetches historical monthly trend data for all brands and normalizes it.
    Kaggle-Safe: Uses external retry logic and robust string slicing.
    """
    # 1. Define the exact modifier and its length
    modifier = " dental"
    n_chars = len(modifier)

    pytrends = TrendReq(hl='en-US', tz=360)

    # 2. Append modifier safely to the search terms
    search_terms = [f"{brand}{modifier}" for brand in unique_brands]
    batch_size = 4
    batches = [search_terms[i:i + batch_size] for i in range(0, len(search_terms), batch_size)]
    historical_dfs = []

    print(f"Fetching history for {len(unique_brands)} brands across {len(batches)} batches...")

    for i, batch in enumerate(batches):
        payload = [anchor_term] + batch

        max_attempts = 4
        for attempt in range(max_attempts):
            try:
                if attempt == 0:
                    print(f"   [Batch {i+1}/{len(batches)}] Fetching {batch}...")
                else:
                    print(f"   [Batch {i+1}/{len(batches)}] Rate limit hit. Retry {attempt}/{max_attempts - 1}...")

                pytrends.build_payload(payload, cat=0, timeframe=timeframe, geo='', gprop='')
                trend_df = pytrends.interest_over_time()

                if not trend_df.empty:
                    if 'isPartial' in trend_df.columns:
                        trend_df = trend_df.drop(columns=['isPartial'])

                    normalized_batch = pd.DataFrame(index=trend_df.index)
                    for term in batch:
                        # Debug: Check if anchor_term or term are missing
                        if anchor_term not in trend_df.columns:
                            print(f"      [Error] Anchor term '{anchor_term}' missing from response!")
                        normalized_batch[term] = trend_df[term] / (trend_df[anchor_term] + 1e-6)

                    historical_dfs.append(normalized_batch)
                else:
                    print(f"      [Warning] Batch {i+1} returned an empty DataFrame.")

                time.sleep(random.uniform(6, 12))
                break

            except Exception as e:
                print(f"      [Exception] Attempt {attempt} failed with error: {e}")
                if attempt < max_attempts - 1:
                    time.sleep(60)
                else:
                    print(f"      [Final Failure] Batch {i+1} abandoned.")

    if historical_dfs:
        master_trend_matrix = pd.concat(historical_dfs, axis=1)

        # 3. ROBUST CLEANUP: Slice exactly n_chars from the END of the column name
        original_cols = list(master_trend_matrix.columns)
        master_trend_matrix.columns = [
            col[:-n_chars] if col.endswith(modifier) else col
            for col in master_trend_matrix.columns
        ]
        print(f"Successfully built matrix. Sample columns (Cleaned): {list(master_trend_matrix.columns)[:5]}")

        return master_trend_matrix
    else:
        print("[Error] No historical trend data was collected. check API/Internet/Terms.")
        return pd.DataFrame()


def apply_pre_launch_trend_score(df, master_trend_matrix, id_col='product_code_base', date_col='first_sale', brand_col='brand_name'):
    """
    Maps the historical trend data to the specific 12 months prior to each product's launch.
    """
    print("Mapping pre-launch trend scores to the dataset...")

    # Ensure dates are datetime objects
    df[date_col] = pd.to_datetime(df[date_col])

    pre_launch_scores = []

    for idx, row in df.iterrows():
        brand = row[brand_col]
        launch_date = row[date_col]

        # We only want data from strictly BEFORE the launch (Data Leakage Protection)
        start_window = launch_date - pd.DateOffset(months=12)
        end_window = launch_date - pd.DateOffset(days=1)

        if brand in master_trend_matrix.columns:
            # Slice the history to just the 12 months before this specific product launched
            brand_history = master_trend_matrix[brand]
            pre_launch_window = brand_history.loc[start_window:end_window]

            # Calculate the average hype during that specific window
            score = pre_launch_window.mean()
        else:
            print("Fallback: brand data failed to pull: score = 0.0")
            score = 0.0 # Fallback if brand data failed to pull

        pre_launch_scores.append({
            id_col: row[id_col],
            'pre_launch_trend_score': score if pd.notna(score) else 0.0
        })

    # Convert results to dataframe and merge back into the main df
    scores_df = pd.DataFrame(pre_launch_scores)

    # Merge, keeping only the first instance of the base code to avoid duplicates
    scores_df = scores_df.drop_duplicates(subset=[id_col])
    merged_df = df.merge(scores_df, on=id_col, how='left')

    return merged_df

---
## 3. K-Means Clustering & Optimal $K$ Selection
The optimal number of profiles is determined by the majority vote of three cluster validity indices: Davies-Bouldin Index, Silhouette Coefficient, and Calinski-Harabasz Index.

**NOTE:** We used weighted average of the three scores to find the optimal `k`. This is different from the paper. The reasons is that the previous algorithm always chose the `k_min` and there were no consensus between the scores (each suggested a different `optimal_k`)
\#Deviation

In [ ]:
def _evaluate_single_k(k, profiles, random_state):
    kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=25)
    labels = kmeans.fit_predict(profiles)
    db = davies_bouldin_score(profiles, labels)
    sil = silhouette_score(profiles, labels)
    ch = calinski_harabasz_score(profiles, labels)
    return k, db, sil, ch

def find_optimal_k(profiles, k_min=2, k_max=20):
    """Sweeps K in parallel based on Normalized Weekly Profiles."""
    num_cores = multiprocessing.cpu_count()
    print(f"      [Parallel] Sweeping K from {k_min} to {k_max} across {num_cores} cores using Profiles...")

    k_range = list(range(k_min, k_max + 1))
    results = Parallel(n_jobs=-1, verbose=0)(
        delayed(_evaluate_single_k)(k, profiles, RANDOM_STATE) for k in k_range
    )

    db_scores, sil_scores, ch_scores = {}, {}, {}
    for k, db, sil, ch in results:
        db_scores[k], sil_scores[k], ch_scores[k] = db, sil, ch

    best_db = min(db_scores, key=db_scores.get)
    best_sil = max(sil_scores, key=sil_scores.get)
    best_ch = max(ch_scores, key=ch_scores.get)

    votes = [best_db, best_sil, best_ch]
    vote_counts = {k: votes.count(k) for k in set(votes)}
    max_votes = max(vote_counts.values())
    winners = [k for k, v in vote_counts.items() if v == max_votes]

    if len(winners) == 1:
        optimal_k = winners[0]
        print(f"      [K-Selection] Clear Majority: K = {optimal_k}")
    else:
        # NO CONSENSUS (3 DIFFERENT VOTES) -> Trigger Normalized Average
        print(f"      [K-Selection] No Consensus (DB:{best_db}, Sil:{best_sil}, CH:{best_ch}). Applying Normalized Average Score...")

        db_vals = list(db_scores.values())
        sil_vals = list(sil_scores.values())
        ch_vals = list(ch_scores.values())

        consensus = {}
        for k in k_range:
            norm_db = (max(db_vals) - db_scores[k]) / (max(db_vals) - min(db_vals)) if max(db_vals) != min(db_vals) else 0
            norm_sil = (sil_scores[k] - min(sil_vals)) / (max(sil_vals) - min(sil_vals)) if max(sil_vals) != min(sil_vals) else 0
            norm_ch = (ch_scores[k] - min(ch_vals)) / (max(ch_vals) - min(ch_vals)) if max(ch_vals) != min(ch_vals) else 0

            consensus[k] = (norm_db + norm_sil + norm_ch) / 3

        optimal_k = max(consensus, key=consensus.get)
        print(f"      [K-Selection] Tie Broken by Consensus Score: K = {optimal_k}")

    return optimal_k

def cluster_profiles(profiles, k):
    """Assigns products to the optimal profiles using K-Means."""
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=25)
    cluster_labels = kmeans.fit_predict(profiles)
    result_df = profiles.copy()
    result_df['cluster'] = cluster_labels
    return result_df, kmeans

---
## 4. Model Training: RF & QRF
Train the machine learning models. The Random Forest classification model predicts the profile shape based on pre-launch features , while the Quantile Regression Forest predicts the total sales volume and confidence bounds.

In [ ]:
def build_preprocessor(X):
    """Encodes categorical features numerically. No scaling required for trees."""
    cat_cols = X.select_dtypes(include=['object', 'category']).columns

    return ColumnTransformer(
        transformers=[
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
        ],
        remainder='passthrough'
    )

def qrf_pinball_scorer(estimator, X, y_true):
    quantiles = [0.05, 0.5, 0.95]
    preds = estimator.predict(X, quantiles=quantiles)
    y_true_arr = np.array(y_true)

    total_loss = 0
    for i, q in enumerate(quantiles):
        err = y_true_arr - preds[:, i]
        loss = np.maximum(q * err, (q - 1) * err)
        total_loss += np.mean(loss)

    return -total_loss

def train_demand_forest(X_train, y_profile_train, y_demand_train):
    preprocessor = build_preprocessor(X_train)
    X_train_processed = preprocessor.fit_transform(X_train)
    tscv = TimeSeriesSplit(n_splits=2)

    print(f"      [{time.strftime('%H:%M:%S')}] Initiating Broad Grid Search for Profile Classifier...")
    rf_grid = {
        'n_estimators': [500],
        'max_depth': [5, 10, 20, 30, None],
        'min_samples_leaf': [1, 2, 4, 8],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2', None, 0.5],
        'class_weight': ['balanced', 'balanced_subsample'],
        'criterion': ['gini', 'entropy']
    }

    kappa_scorer = make_scorer(cohen_kappa_score)
    rf_base = RandomForestClassifier(random_state=RANDOM_STATE)
    rf_cv = GridSearchCV(rf_base, param_grid=rf_grid, scoring=kappa_scorer, cv=tscv, n_jobs=-1, verbose=1)

    rf_cv.fit(X_train_processed, y_profile_train)

    best_rf_params = rf_cv.best_params_.copy()
    best_rf_params['n_estimators'] = N_TREES
    rf_clf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **best_rf_params)
    rf_clf.fit(X_train_processed, y_profile_train)

    print(f"\n      [{time.strftime('%H:%M:%S')}] Initiating Broad Grid Search for Quantile Regressor...")
    qrf_grid = {
        'n_estimators': [500],
        'max_depth': [10, 20, 30, None],
        'min_samples_leaf': [2, 5, 10, 20],
        'min_samples_split': [5, 10, 20, 40],
        'max_features': ['sqrt', 0.5, 0.8, 1.0]
    }

    qrf_base = RandomForestQuantileRegressor(random_state=RANDOM_STATE)
    qrf_cv = GridSearchCV(qrf_base, param_grid=qrf_grid, scoring=qrf_pinball_scorer, cv=tscv, n_jobs=-1, verbose=1)

    qrf_cv.fit(X_train_processed, y_demand_train)

    best_qrf_params = qrf_cv.best_params_.copy()
    best_qrf_params['n_estimators'] = N_TREES
    qrf_reg = RandomForestQuantileRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_qrf_params)
    qrf_reg.fit(X_train_processed, y_demand_train)

    best_parameters_dict = {
        'classifier_best_params': rf_cv.best_params_,
        'regressor_best_params': qrf_cv.best_params_
    }

    return preprocessor, rf_clf, qrf_reg, best_parameters_dict

---
## 5. Forecasting & Interval Synthesis
Once the new products arrive, their pre-launch characteristics are fed into the trained models. The predicted total volume $\hat{d}^h_y$ is multiplied element-wise with the predicted demand profile $\hat{p}_y$ to construct the full T-period forecast.

In [ ]:
def fit_theoretical_bounds(empirical_cdf_matrix):
    """Fits Gamma and Log-Normal distributions to the 1st-99th percentiles."""
    n_samples = empirical_cdf_matrix.shape[0]
    gamma_bounds = np.zeros((n_samples, 3))
    lognorm_bounds = np.zeros((n_samples, 3))

    for i in range(n_samples):
        data = empirical_cdf_matrix[i, :]
        # Add small noise to prevent exact zero variance failures in scipy fitting
        data = data + np.random.uniform(1e-6, 1e-5, size=len(data))

        # Gamma Fit
        shape, loc, scale = stats.gamma.fit(data, floc=0)
        gamma_bounds[i, 0] = stats.gamma.ppf(0.05, shape, loc, scale)
        gamma_bounds[i, 1] = stats.gamma.ppf(0.50, shape, loc, scale)
        gamma_bounds[i, 2] = stats.gamma.ppf(0.95, shape, loc, scale)

        # Log-Normal Fit
        s, loc, scale = stats.lognorm.fit(data, floc=0)
        lognorm_bounds[i, 0] = stats.lognorm.ppf(0.05, s, loc, scale)
        lognorm_bounds[i, 1] = stats.lognorm.ppf(0.50, s, loc, scale)
        lognorm_bounds[i, 2] = stats.lognorm.ppf(0.95, s, loc, scale)

    return gamma_bounds, lognorm_bounds

def generate_forecasts(X_test, preprocessor, rf_clf, qrf_reg, kmeans_model):
    X_test_processed = preprocessor.transform(X_test)

    # 1. Predict Profile Cluster
    predicted_clusters = rf_clf.predict(X_test_processed)
    cluster_centers = kmeans_model.cluster_centers_
    predicted_profiles = np.array([cluster_centers[c] for c in predicted_clusters])

    # 2. Predict Empirical Total Demand CDF (1st to 99th percentile)
    full_quantiles = np.arange(0.01, 1.0, 0.01).tolist()
    empirical_cdf = qrf_reg.predict(X_test_processed, quantiles=full_quantiles)

    # Extract baseline empirical bounds (5th, 50th, 95th indices are 4, 49, 94)
    point_demand = empirical_cdf[:, 49]
    emp_lower_demand = empirical_cdf[:, 4]
    emp_upper_demand = empirical_cdf[:, 94]

    # 3. Fit Theoretical Distributions (NEW)
    gamma_bounds, lognorm_bounds = fit_theoretical_bounds(empirical_cdf)

    # 4. Combine Profile with Demands (Returning Empirical + Gamma for comparison)
    forecast_point = np.round(predicted_profiles * point_demand[:, np.newaxis])

    # Empirical Intervals
    f_emp_lower = np.round(predicted_profiles * emp_lower_demand[:, np.newaxis])
    f_emp_upper = np.round(predicted_profiles * emp_upper_demand[:, np.newaxis])

    # Gamma Intervals
    f_gam_lower = np.round(predicted_profiles * gamma_bounds[:, 0][:, np.newaxis])
    f_gam_upper = np.round(predicted_profiles * gamma_bounds[:, 2][:, np.newaxis])

    # Log-Normal Intervals
    f_log_lower = np.round(predicted_profiles * lognorm_bounds[:, 0][:, np.newaxis])
    f_log_upper = np.round(predicted_profiles * lognorm_bounds[:, 2][:, np.newaxis])

    return predicted_clusters, forecast_point, (f_emp_lower, f_emp_upper), (f_gam_lower, f_gam_upper), (f_log_lower, f_log_upper)

---
## 6. Evaluation Metrics
We evaluate the performance of point forecasts and prediction intervals using specific metrics:
* **PICP (Prediction Interval Coverage Probability):** The percentage of actual values falling within the bounds. Target is the interval size (e.g., 90%).
* **PINAW (Prediction Interval Normalized Average Width):** The average width of the interval relative to the range of actual data.

In [ ]:
def wmape(actual, predicted):
    actual = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    total = actual.sum()
    return np.nan if total == 0 else np.abs(actual - predicted).sum() / total

def evaluate_forecast(actual_matrix, forecast_point_matrix):
    return np.sqrt(mean_squared_error(actual_matrix, forecast_point_matrix))

def winkler_score(actual, lower, upper, alpha=0.10):
    """Calculates the Average Winkler Score for prediction intervals."""
    actual = np.array(actual)
    lower = np.array(lower)
    upper = np.array(upper)

    width = upper - lower
    lower_penalty = np.where(actual < lower, (2 / alpha) * (lower - actual), 0)
    upper_penalty = np.where(actual > upper, (2 / alpha) * (actual - upper), 0)

    return np.mean(width + lower_penalty + upper_penalty)

def evaluate_intervals(actual, lower, upper, alpha=0.10):
    """Calculates PICP, PINAW, and Winkler Score."""
    actual = np.array(actual)
    lower = np.array(lower)
    upper = np.array(upper)

    # 1. PICP (Coverage)
    coverage = ((actual >= lower) & (actual <= upper)).astype(int)
    picp = np.mean(coverage)

    # 2. PINAW (Normalized Width)
    range_target = np.max(actual) - np.min(actual)
    pinaw = np.mean(upper - lower) / range_target if range_target != 0 else np.nan

    # 3. Winkler Score (Rigorous Interval Penalty)
    winkler = winkler_score(actual, lower, upper, alpha)

    return picp, pinaw, winkler

---
## 7. Execution Pipeline Wrapper

In [ ]:
def extract_split_features(df_split, feature_cols, id_col='product_code_base'):
    """
    Dynamically generates split-localized features for products, ensuring price_index
    only averages non-zero transactions, and holiday fields don't compound on multiple sales.
    """
    # 1. Extract static, structural product elements
    static_cols = [c for c in feature_cols if c not in ['price_index', 'holiday']]
    static_cols = [c for c in static_cols if c in df_split.columns]

    # Use drop_duplicates on the subset of static columns to avoid multiple rows per ID
    X_features = df_split[[id_col] + static_cols].drop_duplicates(subset=[id_col]).set_index(id_col)

    # 2. Dynamic Price Index: Average of adj_unit_gross strictly for rows with quantity > 0
    price_df = df_split[df_split['item_quantity'] > 0]
    if len(price_df) > 0:
        price_agg = price_df.groupby(id_col)['adj_unit_gross'].mean().rename('price_index')
    else:
        price_agg = pd.Series(0.0, index=X_features.index, name='price_index')

    # 3. Dynamic Holiday: Total unique holiday days within the horizon weeks
    if 'holiday' in df_split.columns:
        holiday_daily = df_split.groupby([id_col, 'invoice_date'])['holiday'].max().reset_index()
        holiday_agg = holiday_daily.groupby(id_col)['holiday'].sum().rename('holiday')
    else:
        holiday_agg = pd.Series(0, index=X_features.index, name='holiday')

    # Merge dynamically engineered fields back onto the product mapping
    X_features = X_features.join(price_agg, how='left').join(holiday_agg, how='left')
    X_features['price_index'] = X_features['price_index'].fillna(0.0)
    X_features['holiday'] = X_features['holiday'].fillna(0).astype(int)

    return X_features[feature_cols]


def run_demandforest_pipeline(df_train, df_test, feature_cols, id_col='product_code_base', time_col='week_num', qty_col='item_quantity', k_min=2, k_max=8, outlier_quantile=0.95):

    print("   [STEP 1/5] Extracting Normalized Demand Profiles...")
    train_profiles, train_total = extract_demand_profiles(df_train, id_col, time_col, qty_col)

    # TRAINING OUTLIER REMOVAL
    demand_threshold = train_total['total_demand_h'].quantile(outlier_quantile)
    valid_train_products = train_total[train_total['total_demand_h'] <= demand_threshold][id_col]

    train_profiles = train_profiles.loc[train_profiles.index.isin(valid_train_products)]
    train_total = train_total[train_total[id_col].isin(valid_train_products)]


    # IF YOU WANT TO RETRIEVE GOOGLE TREND SCORES FOR THE FIRST TIME

    print("   [STEP 0] Pre-processing Features (Google Trends & Dates)...")

    # 1. Extract unique brands & dynamically calculate the timeframe
    all_brands = pd.concat([df_train['brand_name'], df_test['brand_name']]).dropna().unique().tolist()
    df_train['first_sale'] = pd.to_datetime(df_train['first_sale'])
    df_test['first_sale'] = pd.to_datetime(df_test['first_sale'])

    all_dates = pd.concat([df_train['first_sale'], df_test['first_sale']])
    start_date = all_dates.min() - pd.DateOffset(months=12)
    dynamic_timeframe = f"{start_date.strftime('%Y-%m-%d')} {all_dates.max().strftime('%Y-%m-%d')}"

    # 2. KAGGLE SMART CACHING LOGIC
    cached_trends_path = '/kaggle/input/dental-brand-trends/master_brand_trends.csv'
    working_trends_path = '/kaggle/working/master_brand_trends.csv'

    if os.path.exists(cached_trends_path):
        print("      [Google Trends] Cached file found! Loading instantly from Kaggle Inputs...")
        master_trends = pd.read_csv(cached_trends_path, index_col=0, parse_dates=True)

    elif os.path.exists(working_trends_path):
        print("      [Google Trends] Loading from current working session...")
        master_trends = pd.read_csv(working_trends_path, index_col=0, parse_dates=True)

    else:
        print(f"      [Google Trends] No cache found. Fetching from API: {dynamic_timeframe}")
        master_trends = fetch_historical_trend_matrix(
            unique_brands=all_brands,
            anchor_term="dental products",
            timeframe=dynamic_timeframe
        )
        # Save the result immediately so you can download it from the 'Output' tab later
        if not master_trends.empty:
            master_trends.to_csv(working_trends_path)
            print(f"      [Google Trends] Successfully saved trend data to {working_trends_path}")

    # 3. Apply the rolling window logic to the main dataframes
    if not master_trends.empty:
        df_train = apply_pre_launch_trend_score(df_train, master_trends)
        df_test = apply_pre_launch_trend_score(df_test, master_trends)
    else:
        print("      [Warning] No trend data available to merge. Filling with 0.0")
        df_train['pre_launch_trend_score'] = 0.0
        df_test['pre_launch_trend_score'] = 0.0



    # IF YOU ALREADY HAVE THE GOOGLE TREND SCORES CSV FILE

    # ====================================================================
    # CONFIGURATION: Set the path to your downloaded/uploaded CSV file
    # ====================================================================

    # google_trends_csv_path = '/kaggle/input/dental-brand-trends/master_brand_trends.csv'

    # print("   [STEP 0] Pre-processing Features (Loading Google Trends CSV)...")

    # if os.path.exists(google_trends_csv_path):
    #     print(f"      [Google Trends] Loading data directly from: {google_trends_csv_path}")

    #     # 1. Load the CSV and force the index to be strict datetime objects
    #     master_trends = pd.read_csv(google_trends_csv_path, index_col=0, parse_dates=True)

    #     # 2. ROBUST CLEANUP: Safely slice " dental" from the end of the column names
    #     # to guarantee a perfect match with the 'brand' column in your dataset.
    #     modifier = " dental"
    #     n_chars = len(modifier)
    #     master_trends.columns = [
    #         col[:-n_chars] if col.endswith(modifier) else col
    #         for col in master_trends.columns
    #     ]

    #     # 3. STRICT DATE FORMATTING: Ensure your main dataframes use exact datetimes
    #     # (Using dayfirst=True to protect against EU vs US format confusion)
    #     df_train['first_sale'] = pd.to_datetime(df_train['first_sale'], dayfirst=True, errors='coerce')
    #     df_test['first_sale'] = pd.to_datetime(df_test['first_sale'], dayfirst=True, errors='coerce')

    #     # 4. Apply the mapping function
    #     df_train = apply_pre_launch_trend_score(df_train, master_trends)
    #     df_test = apply_pre_launch_trend_score(df_test, master_trends)

    # else:
    #     # Failsafe if the file path is incorrect
    #     print(f"      [ERROR] Could not find the Google Trends file at: {google_trends_csv_path}")
    #     print("      [Warning] Filling pre_launch_trend_score with 0.0")
    #     df_train['pre_launch_trend_score'] = 0.0
    #     df_test['pre_launch_trend_score'] = 0.0


    X_train_raw = extract_split_features(df_train, feature_cols, id_col)
    common_train_idx = train_profiles.index.intersection(X_train_raw.index)

    # CHRONOLOGICAL SORTING FOR TIMESERIESSPLIT
    product_launch_dates = df_train.groupby(id_col)['first_sale'].min()
    sorted_common_idx = product_launch_dates.loc[common_train_idx].sort_values().index

    train_profiles = train_profiles.loc[sorted_common_idx]
    X_train = X_train_raw.loc[sorted_common_idx]
    y_demand_train = train_total.set_index(id_col).loc[sorted_common_idx]['total_demand_h']

    print("   [STEP 2/5] Initiating Parallel K-Means Clustering Sweep...")
    opt_k = find_optimal_k(train_profiles, k_min=k_min, k_max=k_max)
    clustered_profiles, kmeans_model = cluster_profiles(train_profiles, opt_k)
    y_profile_train = clustered_profiles['cluster']

    print(f"   [STEP 3/5] Training DemandForest Models (Optimal K={opt_k})...")
    preprocessor, rf_clf, qrf_reg, best_params = train_demand_forest(X_train, y_profile_train, y_demand_train)

    print("   [STEP 4/5] Preparing Test Data & Generating Cold-Start Forecasts...")
    test_profiles, test_total = extract_demand_profiles(df_test, id_col, time_col, qty_col)
    X_test_raw = extract_split_features(df_test, feature_cols, id_col)
    common_test_idx = test_profiles.index.intersection(X_test_raw.index)

    test_profiles = test_profiles.loc[common_test_idx]
    X_test = X_test_raw.loc[common_test_idx]

    pred_clusters, f_point, emp_bounds, gam_bounds, log_bounds = generate_forecasts(
        X_test, preprocessor, rf_clf, qrf_reg, kmeans_model
    )

    print("   [STEP 5/5] Evaluating Forecast Accuracy & Synthesizing Results...")
    actual_matrix = test_profiles.values * test_total.set_index(id_col).loc[test_profiles.index]['total_demand_h'].values[:, np.newaxis]
    actual_totals = test_total.set_index(id_col).loc[test_profiles.index]['total_demand_h'].values
    test_true_clusters = kmeans_model.predict(test_profiles)

    rmse = evaluate_forecast(actual_matrix, f_point)

    # Evaluate Empirical, Gamma, and Log-Normal bounds (Now unpacks 3 variables)
    picp_emp, pinaw_emp, wink_emp = evaluate_intervals(actual_matrix, emp_bounds[0], emp_bounds[1])
    picp_gam, pinaw_gam, wink_gam = evaluate_intervals(actual_matrix, gam_bounds[0], gam_bounds[1])
    picp_log, pinaw_log, wink_log = evaluate_intervals(actual_matrix, log_bounds[0], log_bounds[1])

    pred_totals = f_point.sum(axis=1)
    wmape_score = wmape(actual_totals, pred_totals)
    kappa = cohen_kappa_score(test_true_clusters, pred_clusters)
    accuracy = accuracy_score(test_true_clusters, pred_clusters)

    # =================================================================
    # NEW: PERMUTATION FEATURE IMPORTANCE
    # =================================================================
    print("   [STEP 6/6] Calculating Permutation Feature Importance...")
    X_test_processed = preprocessor.transform(X_test)

    clf_importance = permutation_importance(rf_clf, X_test_processed, test_true_clusters, n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
    reg_importance = permutation_importance(qrf_reg, X_test_processed, actual_totals, n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)

    feature_importance_dict = {
        'features': feature_cols,
        'classifier_importance': clf_importance.importances_mean.tolist(),
        'regressor_importance': reg_importance.importances_mean.tolist()
    }
    # =================================================================

    print(f"Prediction Interval (Empirical 90%) : PICP = {picp_emp*100:.1f}%, Winkler = {wink_emp:.2f}")
    print(f"Prediction Interval (Gamma 90%)     : PICP = {picp_gam*100:.1f}%, Winkler = {wink_gam:.2f}")

    pipeline_metrics_dict = {
        'optimal_k': int(opt_k),
        'profile_classifier_kappa': float(kappa),
        'profile_classifier_accuracy': float(accuracy),
        'forecast_horizon_rmse': float(rmse),
        'total_horizon_wmape': float(wmape_score),
        'emp_interval_picp_90': float(picp_emp),
        'emp_interval_pinaw_90': float(pinaw_emp),
        'emp_interval_winkler_90': float(wink_emp),
        'gam_interval_picp_90': float(picp_gam),
        'gam_interval_pinaw_90': float(pinaw_gam),
        'gam_interval_winkler_90': float(wink_gam),
        'tuned_hyperparameters': best_params,
        'feature_importance': feature_importance_dict
    }

    return f_point, gam_bounds[0], gam_bounds[1], actual_totals, test_profiles.index, pipeline_metrics_dict

---
## 8. Temporal Splitting
To ensure a realistic cold-start evaluation, we split the product pool chronologically based on their `first_sale` timestamp. This guarantees that the models (RF and QRF) are only trained on products that existed *before* the new products were launched.

In [ ]:
def _split_timestamps(df_full, id_col='product_code_base', time_col='first_sale', test_share=0.2, val_share_of_train=0.2):
    """Calculates chronological cutoff dates (test_ts, validation_ts) based on product launch sequence."""
    products = df_full[[id_col, time_col]].drop_duplicates().sort_values(time_col)
    n_total = len(products)

    if n_total == 0:
        return pd.Timestamp.max, pd.Timestamp.max

    # Calculate Test Set cutoff timestamp
    test_idx = int(n_total * (1 - test_share))
    test_idx = max(0, min(test_idx, n_total - 1))
    test_ts = products.iloc[test_idx][time_col]

    # Calculate Validation Set cutoff from the remaining pool
    train_val = products[products[time_col] < test_ts]
    n_train_val = len(train_val)

    if n_train_val > 0:
        val_idx = int(n_train_val * (1 - val_share_of_train))
        val_idx = max(0, min(val_idx, n_train_val - 1))
        validation_ts = train_val.iloc[val_idx][time_col]
    else:
        validation_ts = test_ts

    return test_ts, validation_ts


def prepare_demandforest_data(df, cols_to_drop):
    """Cleans data, targets unusual sales anomalies, and flags categories."""
    df_clean = df.copy()

    # Aggregate by product_code_base and find the minimum first_sale
    if 'product_code_base' in df_clean.columns:
        df_clean['first_sale'] = df_clean.groupby('product_code_base')['first_sale'].transform('min')

    # FIX: Overwrite quantity and prices for rows flagged as unusual sales anomalies
    if 'unusual_sales' in df_clean.columns:
        mask_unusual = df_clean['unusual_sales']==1
        df_clean.loc[mask_unusual, 'item_quantity'] = 0
        for p_col in ['unit_gross', 'adj_unit_gross']:
            if p_col in df_clean.columns:
                df_clean.loc[mask_unusual, p_col] = 0.0

    # Calculate relative week number (1 to T)
    if 'week_num' not in df_clean.columns:
        df_clean['week_num'] = (df_clean['invoice_date'] - df_clean['first_sale']).dt.days // 7 + 1

    # Filter strictly to the forecast horizon
    df_clean = df_clean[df_clean['week_num'] <= FORECAST_HORIZON_WEEKS]

    # Drop specified redundant base arrays
    df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')

    return df_clean

---
## 9. Execution Wrapper
This wrapper maps all sub-components: it splits the data, runs the validation phase (Training on Train $\rightarrow$ Predicting on Val), optionally runs the testing phase (Training on Train+Val $\rightarrow$ Predicting on Test), and exports the synthesized forecast intervals.

---
## 10. Execution

In [ ]:
def wrapper_demandforest(
    df, feature_cols, cols_to_drop,
    id_col='product_code_base', time_col='week_num', qty_col='item_quantity',
    test_share=0.2, val_share_of_train=0.2, k_min=2, k_max=8, flag_export=False
):
    df_clean = prepare_demandforest_data(df, cols_to_drop)
    max_date = df_clean['invoice_date'].max()

    test_ts, validation_ts = _split_timestamps(
        df_clean, id_col=id_col, time_col='first_sale',
        test_share=test_share, val_share_of_train=val_share_of_train
    )
    hz_buffer = pd.Timedelta(weeks=FORECAST_HORIZON_WEEKS)

    print("\nPHASE 1: VALIDATION")
    df_train = df_clean[df_clean['first_sale'] <= validation_ts - hz_buffer].copy()
    df_val = df_clean[(df_clean['first_sale'] >= validation_ts) & (df_clean['first_sale'] <= test_ts - hz_buffer)].copy()

    val_point, val_lower, val_upper, val_actuals, val_idx, val_metrics = run_demandforest_pipeline(
        df_train, df_val, feature_cols, id_col, time_col, qty_col, k_min=k_min, k_max=k_max
    )

    val_results_df = pd.DataFrame({
        'product_code_base': val_idx,
        'actual_total_hz_qty': val_actuals,
        'forecast_point_total': val_point.sum(axis=1),
        'forecast_lower_90_total': np.maximum(val_lower.sum(axis=1), 1),
        'forecast_upper_90_total': val_upper.sum(axis=1)
    })

    print("\nPHASE 2: TESTING")
    df_train_val = df_clean[df_clean['first_sale'] <= test_ts - hz_buffer].copy()
    df_test = df_clean[(df_clean['first_sale'] >= test_ts) & (df_clean['first_sale'] <= max_date - hz_buffer)].copy()

    test_point, test_lower, test_upper, test_actuals, test_idx, test_metrics = run_demandforest_pipeline(
        df_train_val, df_test, feature_cols, id_col, time_col, qty_col, k_min=k_min, k_max=k_max
    )

    test_results_df = pd.DataFrame({
        'product_code_base': test_idx,
        'actual_total_hz_qty': test_actuals,
        'forecast_point_total': test_point.sum(axis=1),
        'forecast_lower_90_total': np.maximum(test_lower.sum(axis=1), 1),
        'forecast_upper_90_total': test_upper.sum(axis=1)
    })

    # Consolidate and Export Metrics
    metrics_summary = pd.DataFrame({
        'Metric': list(val_metrics.keys()),
        'Validation': list(val_metrics.values()),
        'Test': list(test_metrics.values())
    })

    print("\n--- FINAL PIPELINE METRICS ---")
    print(metrics_summary.to_string(index=False))

    if flag_export:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        val_results_df.to_excel(f'DemandForest_Validation_Results_{timestamp}.xlsx', index=False)
        test_results_df.to_excel(f'DemandForest_Test_Results_{timestamp}.xlsx', index=False)
        metrics_summary.to_excel(f'DemandForest_Metrics_{timestamp}.xlsx', index=False)
        print(f"\n[EXPORT SUCCESS] Files saved with timestamp {timestamp}")

    master_run_summary = {
        'metadata': {'forecast_horizon_weeks': int(FORECAST_HORIZON_WEEKS)},
        'validation_phase_results': val_metrics,
        'test_phase_results': test_metrics
    }

    return val_results_df, test_results_df, master_run_summary

---
## 11. Global Parameters & Execution
This cell loads the dataset, defines the pre-launch features ($f_z$), and triggers the pipeline. Note that in DemandForest, we **strictly** pass static, pre-launch characteristics to `PRE_LAUNCH_FEATURES` (e.g., category, price index, substitutability). We do *not* pass post-launch behavioral features (like CV of quantity).

In [ ]:
# ---------------------------------------------------------
# GLOBAL PARAMETERS & EXECUTION
# ---------------------------------------------------------

k_min = 2
k_max = 10

# Columns to drop during initial cleaning
COLS2DROP_BASE = [
    'is_corrected_cq', 'is_corrected_cq_opposite', 'price_level',
    'is_corrected_only_price', 'last_sale', 'sub cat', 'sub sub cat', 'unusual_sales'
]

# The F_z Launch features structure (using product_code_base as key)
PRE_LAUNCH_FEATURES = [
    'brand_name', 'Producer', 'Region', 'Category',
    'full_sub_sub_cat', 'full_sub_cat', 'is_new_brand',
    'num_similars', 'substitutability', 'price_index',
    'holiday', 'pre_launch_trend_score'
]

try:
    # Download and load data logic (kept same as before)
    file_id = "file_id"
    direct_url = f"https://drive.usercontent.google.com/download?export=download&confirm=t&id={file_id}"
    output_filename = "dataset.pkl"

    if not os.path.exists(output_filename):
        print("Downloading file from Google Drive...")
        response = requests.get(direct_url, stream=True)
        if response.status_code == 200:
            total_size = int(response.headers.get('content-length', 0))
            with open(output_filename, "wb") as f, tqdm(desc=output_filename, total=total_size, unit="B", unit_scale=True) as bar:
                for chunk in response.iter_content(chunk_size=32768):
                    if chunk: f.write(chunk); bar.update(len(chunk))
        else:
            raise Exception("Download failed.")

    print("Loading pickle file...")
    df = pd.read_pickle(output_filename)

    # Trigger pipeline using product_code_base
    val_results_df, test_results_df, summary_info = wrapper_demandforest(
        df=df,
        feature_cols=PRE_LAUNCH_FEATURES,
        cols_to_drop=COLS2DROP_BASE,
        id_col='product_code_base',
        time_col='week_num',
        qty_col='item_quantity',
        test_share=test_share,
        val_share_of_train=val_share_of_train,
        k_min=k_min,
        k_max=k_max,
        flag_export=True
    )

    print("\n[PIPELINE SUMMARY SUCCESS]")

except Exception as e:
    print(f"Error: {e}")
